In [4]:
from tensordict import TensorDict


TensorDict.load_memmap('/Users/yawerijaz/figgie_gym/data/supervised_data_tensordict')

TensorDict(
    fields={
        cash: MemoryMappedTensor(shape=torch.Size([2050000]), device=cpu, dtype=torch.float32, is_shared=True),
        hidden.agent: NonTensorStack(
            ['agent_0', 'agent_1', 'agent_2', 'agent_3', 'agen...,
            batch_size=torch.Size([2050000]),
            device=cpu),
        hidden.agent_type: NonTensorStack(
            ['NoiseAgent', 'NoiseAgent', 'NoiseAgent', 'NoiseA...,
            batch_size=torch.Size([2050000]),
            device=cpu),
        hidden.experiment: MemoryMappedTensor(shape=torch.Size([2050000]), device=cpu, dtype=torch.int64, is_shared=True),
        hidden.goal_suit: NonTensorStack(
            ['Spade', 'Spade', 'Spade', 'Spade', 'Spade', 'Spa...,
            batch_size=torch.Size([2050000]),
            device=cpu),
        hidden.goal_suit_code: MemoryMappedTensor(shape=torch.Size([2050000]), device=cpu, dtype=torch.int64, is_shared=True),
        hidden.num_cardcounter_agents: MemoryMappedTensor(shape=torch.Size([

In [6]:
from figgie_gym.models.tensordict_dataloader import TensorDictDataModule
datamod = TensorDictDataModule("/Users/yawerijaz/figgie_gym/data/supervised_data_tensordict_dev")
datamod.setup()

for z in (datamod.train_dataloader()):
    break

In [ ]:
from tensordict import stack


z['x']
# 'step', 'steps_remaining', 'remaining_game_portion', 'cash', 
# per_suit: 'known_count', 'bid_price', 'bid_quantity', 'ask_price', 'ask_quantity', 'last_price', 'volume', 'self_position'
# per_suit - agent_trade_summaries: 'buy_quantity', 'buy_consideration', 'sell_quantity', 'sell_consideration', 'min_net_quantity_change'

stack([z['x', 'per_suit', 'agent_trade_summaries'][k] for k in ['buy_quantity', 'buy_consideration', 'sell_quantity', 'sell_consideration', 'min_net_quantity_change']], dim=-1)

tensor([[[-1, -1, -1, -2,  0],
         [-2, -3,  0,  0, -1],
         [ 0, -2, -4, -2, -1],
         [-2,  0, -2, -2, -2]],

        [[ 0, -1,  0, -1,  0],
         [-1, -1,  0,  0, -1],
         [ 0, -1, -1, -2,  0],
         [ 0,  0, -2,  0, -2]],

        [[-1, -1, -1, -2,  0],
         [-2, -3,  0,  0, -1],
         [ 0, -2, -1, -2, -1],
         [-2,  0, -2, -2, -2]],

        ...,

        [[-1, -1, -2, -1,  0],
         [ 0, -3,  0, -2, -1],
         [-1, -2, -2,  0, -1],
         [-2,  0, -2, -2, -2]],

        [[-2, -1, -1, -1,  0],
         [ 0, -3,  0, -3, -1],
         [-2, -2, -4,  0, -1],
         [-2,  0, -2, -2, -2]],

        [[-2, -1, -1, -1,  0],
         [ 0, -3,  0, -2, -1],
         [-2, -2, -4,  0, -1],
         [-2,  0, -2, -2, -2]]])

In [44]:
['step',
 'steps_remaining',
 'remaining_game_portion',
 'cash',
 ]

['step', 'steps_remaining', 'remaining_game_portion', 'cash']

In [45]:
list(td["per_suit"].keys())
[
    'suit_code',
    'known_count',
    'bid_price',
    'bid_quantity',
    'ask_price',
    'ask_quantity',
    'last_price',
    'volume',
    'self_position',
 ]

['suit_code',
 'known_count',
 'bid_price',
 'bid_quantity',
 'ask_price',
 'ask_quantity',
 'last_price',
 'volume',
 'self_position']

In [ ]:
suit_stacked = stack(
    [
        td["per_suit", "suit_code"],
        td["per_suit", "known_count"],
    ],
    dim=-1,
)
suit_stacked.flatten(-2)

tensor([[0, 2, 1, 3, 2, 4, 3, 5],
        [0, 3, 1, 2, 2, 5, 3, 3],
        [0, 3, 1, 3, 2, 3, 3, 5]])

In [47]:
summary_stacked = stack(
    [
        td["per_suit", "agent_trade_summaries", "buy_consideration"],
        td["per_suit", "agent_trade_summaries", "sell_consideration"],
    ],
    dim=-1,
)
summary_stacked.shape

torch.Size([3, 4, 5, 2])

In [3]:
from torch import nn
import torch

# w = nn.Linear(2, 2)
# with torch.no_grad():
#     w.weight.copy_( torch.diag(torch.Tensor([3,2])))
#     w.bias.copy_(torch.zeros(2))

# summary_out = w(summary_stacked.to(dtype=torch.float32))

In [4]:
simple_suit_features = stack(
    [
        td["per_suit", "suit_code"],
        td["per_suit", "known_count"],
    ],
    dim=-1,
).to(dtype=torch.float32)

suit_agent_features = stack(
    [
        td["per_suit", "agent_trade_summaries", "buy_consideration"],
        td["per_suit", "agent_trade_summaries", "sell_consideration"],
    ],
    dim=-1,
).to(dtype=torch.float32)

suit_agent_embedding = nn.Sequential(
    nn.Linear(2, 2)
)

suit_agent_embedding_out = suit_agent_embedding(suit_agent_features)
torch.concat([simple_suit_features, suit_agent_embedding_out.flatten(-2)], dim=-1).flatten(-2).shape

torch.Size([3, 48])

In [42]:
summary_out.flatten(-2).shape

torch.Size([20500, 4, 10])

In [21]:
torch.concat([suit_stacked, summary_out.flatten(-2)], dim=-1).shape

torch.Size([20500, 4, 12])

In [19]:
import numpy as np

In [79]:
a, b = np.sort(np.random.randint(0, 5, size=2))
a, (b-a), 5-b

(np.int64(2), np.int64(2), np.int64(1))